In [1]:
!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install -q langchain langchain-huggingface langgraph
!pip install -q supabase pymupdf openai-whisper
!pip install -q fastapi uvicorn pyngrok python-multipart httpx
!pip install -q firebase-admin

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 10.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.2.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.0/730.0 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

In [2]:
!pip install -q huggingface-hub>=1.3.0 --upgrade

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-huggingface 1.2.0 requires huggingface-hub<1.0.0,>=0.33.4, but you have huggingface-hub 1.4.1 which is incompatible.


In [3]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

GPU available: True
GPU name: Tesla T4
VRAM: 15.64 GB


In [4]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

hf_token = secrets.get_secret("HF_TOKEN")
supabase_url = secrets.get_secret("SUPABASE_URL")
supabase_key = secrets.get_secret("SUPABASE_KEY")

print("All tokens loaded!")

All tokens loaded!


**Downloading Medgemma Model**

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "google/medgemma-4b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)
print("MedGemma 4B loaded successfully!")

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

MedGemma 4B loaded successfully!


In [6]:
!pip install -q -U bitsandbytes>=0.46.1

**Accessing downloaded Medgemma Model**

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

save_path = "/kaggle/input/datasets/naveen2107/medgemma-4b-saved/medgemma-4b-saved"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(save_path)
model = AutoModelForCausalLM.from_pretrained(
    save_path,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded from dataset!")

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:250: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Model loaded from dataset!


**creating langchain Runnable with downloaded model**

In [8]:

from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.3,
    do_sample=True
)

llm = HuggingFacePipeline(pipeline=pipe)
medgemma = ChatHuggingFace(llm=llm)

print("LangChain ready")

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LangChain ready


**Agent Creation**

In [9]:
import re
from langgraph.graph import StateGraph, END
from langchain_core.prompts import PromptTemplate
from typing import TypedDict, Optional
import json

# Defining what data flows through the agent
class PatientState(TypedDict):
    patient_id: str
    wearable: dict
    symptoms: dict
    labs: dict
    risk_score: Optional[int]
    risk_level: Optional[str]
    action_card: Optional[dict]
    reasoning: Optional[str]

# The main prompt
FUSION_PROMPT = PromptTemplate.from_template("""
You are JointGuard, an RA flare prediction AI.

PATIENT DATA:
Sleep: {sleep_hrs} hours
HRV: {hrv} ms
Steps: {steps}
Pain score: {pain}/10
Morning stiffness: {stiffness} minutes
Fatigue: {fatigue}
CRP: {crp} mg/L
ESR: {esr} mm/hr

Return ONLY valid JSON, nothing else:
{{
  "flare_risk_score": <number 0-100>,
  "risk_level": "<GREEN or AMBER or RED>",
  "key_drivers": ["reason1", "reason2", "reason3"],
  "action_card": {{
    "rest": "instruction here",
    "medication": "instruction here",
    "doctor_contact": "instruction or null"
  }},
  "reasoning": "2 sentence explanation"
}}
""")

# Node — assess risk

def assess_risk(state: PatientState) -> dict:
    w = state["wearable"]
    s = state["symptoms"]
    l = state["labs"]

    prompt = f"""You are a medical AI. Analyze this RA patient data.

Patient Data:
- Sleep: {w.get("sleep_hours", "N/A")} hours
- HRV: {w.get("hrv_rmssd", "N/A")} ms
- Steps: {w.get("steps", "N/A")}
- Pain score: {s.get("pain_score", "N/A")}/10
- Morning stiffness: {s.get("joint_stiffness_minutes", "N/A")} minutes
- Fatigue: {s.get("fatigue_level", "N/A")}
- CRP: {l.get("CRP_mg_per_L", "unknown")} mg/L
- ESR: {l.get("ESR_mm_per_hr", "unknown")} mm/hr

Return ONLY this JSON:
{{
  "flare_risk_score": <0-100>,
  "risk_level": "<GREEN or AMBER or RED>",
  "key_drivers": ["reason1", "reason2"],
  "action_card": {{
    "rest": "instruction",
    "medication": "instruction",
    "doctor_contact": "null"
  }},
  "reasoning": "2 sentence explanation"
}}"""

    output = pipe(prompt)
    raw = output[0]["generated_text"]

    # Extract FIRST json block only
    match = re.search(r'```json\s*(\{.*?\})\s*```', raw, re.DOTALL)
    if match:
        json_str = match.group(1)
    else:
        # fallback — find first { to last }
        start = raw.find("{")
        end = raw.rfind("}") + 1
        json_str = raw[start:end]

    result = json.loads(json_str)

    return {
        "risk_score": result["flare_risk_score"],
        "risk_level": result["risk_level"],
        "action_card": result["action_card"],
        "reasoning": result["reasoning"]
    }

# Rebuild graph
graph = StateGraph(PatientState)
graph.add_node("assess", assess_risk)
graph.add_edge("assess", END)
graph.set_entry_point("assess")
agent = graph.compile()

print("Agent Ready")

Agent Ready


**Lab PDF Parser**

In [10]:
import fitz  
import json, re

def extract_lab_text(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    return "".join(page.get_text() for page in doc)

def parse_labs(pdf_path: str) -> dict:
    text = extract_lab_text(pdf_path)

    prompt = f"""You are a medical AI. Extract lab values from this report.

Lab Report Text:
{text[:2000]}

Return ONLY this JSON:
{{
  "CRP_mg_per_L": <number or null>,
  "ESR_mm_per_hr": <number or null>,
  "RF_titre": "<string or null>",
  "anti_CCP": "<positive or negative or null>",
  "collection_date": "<YYYY-MM-DD or null>"
}}"""

    output = pipe(prompt)
    raw = output[0]["generated_text"]

    match = re.search(r'```json\s*(\{.*?\})\s*```', raw, re.DOTALL)
    if match:
        json_str = match.group(1)
    else:
        start = raw.find("{")
        end = raw.rfind("}") + 1
        json_str = raw[start:end]

    return json.loads(json_str)

print("Lab parser ready!")

Lab parser ready!


**superbase connection**

In [11]:
from supabase import create_client

sb = create_client(supabase_url, supabase_key)
print("Supabase connected!")

Supabase connected!


In [12]:
def save_labs(patient_id: str, labs: dict):
    sb.table("labs").insert({
        "patient_id": patient_id,
        "crp_mg_per_l": labs.get("CRP_mg_per_L"),
        "esr_mm_per_hr": labs.get("ESR_mm_per_hr"),
        "rf_titre": labs.get("RF_titre"),
        "anti_ccp": labs.get("anti_CCP")
    }).execute()
    print(" Labs saved to Supabase!")

In [13]:
def save_risk(patient_id: str, result: dict):
    sb.table("risk_scores").insert({
        "patient_id": patient_id,
        "risk_score": result.get("risk_score"),
        "risk_level": result.get("risk_level"),
        "action_card": result.get("action_card"),
        "reasoning": result.get("reasoning")
    }).execute()
    print(" Risk score saved to Supabase!")

In [14]:
from datetime import datetime, timedelta
import random

base_date = datetime.now() - timedelta(days=30)

for day in range(30):
    date = base_date + timedelta(days=day)
    flare = max(0, (day - 7) / 4) if 7 < day < 12 else 0

    sb.table("wearable_readings").insert({
        "patient_id": "patient_demo",
        "sleep_hours": round(7.5 - flare * 2 + random.gauss(0, 0.3), 1),
        "hrv_rmssd": round(55 - flare * 20 + random.gauss(0, 3), 1),
        "steps": int(8000 - flare * 4000 + random.gauss(0, 500)),
        "created_at": date.isoformat()
    }).execute()

    sb.table("risk_scores").insert({
        "patient_id": "patient_demo",
        "risk_score": int(20 + flare * 70 + random.gauss(0, 5)),
        "risk_level": "RED" if flare > 0.7 else "AMBER" if flare > 0.3 else "GREEN",
        "reasoning": "Simulated demo data",
        "created_at": date.isoformat()
    }).execute()

print(" 30 days demo data inserted!")

 30 days demo data inserted!


**Demo Inseration**

In [15]:
from datetime import datetime, timedelta
import random

base_date = datetime.now() - timedelta(days=15)

for day in range(15):
    date = base_date + timedelta(days=day)
    flare = max(0, (day - 7) / 4) if 7 < day < 12 else 0
    
    sb.table("risk_scores").insert({
        "patient_id": "patient_001",
        "risk_score": int(20 + flare * 70 + random.gauss(0, 5)),
        "risk_level": "RED" if flare > 0.7 else "AMBER" if flare > 0.3 else "GREEN",
        "reasoning": f"Day {day+1}: CRP elevated, HRV declining",
        "created_at": date.isoformat()
    }).execute()

print("15 days history added for patient_001!")

15 days history added for patient_001!


**FastAPI**

In [16]:
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok
import uvicorn
import threading
from pydantic import BaseModel
import tempfile, os
from datetime import datetime

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"]
)

class PredictRequest(BaseModel):
    patient_id: str
    wearable: dict
    symptoms: dict
    labs: dict

@app.get("/health")
def health():
    return {"status": "running"}

@app.post("/predict")
def predict(req: PredictRequest):
    result = agent.invoke({
        "patient_id": req.patient_id,
        "wearable": req.wearable,
        "symptoms": req.symptoms,
        "labs": req.labs,
        "risk_score": None,
        "risk_level": None,
        "action_card": None,
        "reasoning": None
    })
    sb.table("risk_scores").insert({
        "patient_id": req.patient_id,
        "risk_score": result["risk_score"],
        "risk_level": result["risk_level"],
        "action_card": result["action_card"],
        "reasoning": result["reasoning"],
        "created_at": datetime.now().isoformat()
    }).execute()
    return {
        "risk_score": result["risk_score"],
        "risk_level": result["risk_level"],
        "action_card": result["action_card"],
        "reasoning": result["reasoning"]
    }

@app.post("/upload-lab/{patient_id}")
async def upload_lab(patient_id: str, pdf: UploadFile = File(...)):
    with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as f:
        f.write(await pdf.read())
        path = f.name

    # Use existing parse_labs function
    labs = parse_labs(path)
    os.unlink(path)

    sb.table("labs").insert({
        "patient_id": patient_id,
        "crp_mg_per_l": labs.get("CRP_mg_per_L"),
        "esr_mm_per_hr": labs.get("ESR_mm_per_hr"),
        "rf_titre": labs.get("RF_titre"),
        "anti_ccp": labs.get("anti_CCP"),
        "created_at": datetime.now().isoformat()
    }).execute()

    return labs

# Start server
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run, daemon=True)
thread.start()

# Connect ngrok
ngrok_token = secrets.get_secret("NGROK_TOKEN")
ngrok.set_auth_token(ngrok_token)
public_url = ngrok.connect(8000)
print(f"API running at: {public_url}")

INFO:     Started server process [24]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


API running at: NgrokTunnel: "https://trena-nondrinkable-nongelatinously.ngrok-free.dev" -> "http://localhost:8000"
